# 07 — Final Error Analysis and Holdout Gate Test

This notebook performs the manual audit required after automated RAG evaluation. It answers three questions:

1. Did a low score represent a real answer failure or a phrase-matcher false negative?
2. Was the primary failure caused by retrieval or generation?
3. Does the provisional `0.58` evidence gate reject fresh unsupported questions that were not used to choose it?

In [1]:
from pathlib import Path
import json, os, sys, logging
import numpy as np
import pandas as pd
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
logging.getLogger('tensorflow').setLevel(logging.ERROR)

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

experiment = json.loads((PROJECT_ROOT / 'evaluation/rag_experiment_results.json').read_text(encoding='utf-8'))
manual_review = pd.DataFrame(json.loads((PROJECT_ROOT / 'evaluation/manual_grounded_top3_review.json').read_text(encoding='utf-8')))
all_rows = pd.DataFrame(experiment['rows'])
selected_rows = all_rows[(all_rows.configuration == 'grounded_top3') & (all_rows.supported)].copy()
audit = selected_rows.merge(manual_review, on='question_id', how='inner')
print('Selected supported runs:', len(audit))
print('Configuration:', 'grounded_top3')

Selected supported runs: 10
Configuration: grounded_top3


## Checkpoint 1 — Automated scoring and manual judgment serve different purposes

The accepted-phrase scorer is reproducible but literal. Manual review uses the expected evidence and distinguishes full, partial, and missing concepts. Manual point coverage allows `0.5` credit for a partially expressed expected point; this judgment is documented per question rather than silently changing the automatic rules after seeing outputs.

In [2]:
automatic_coverage = audit.answer_point_coverage.mean()
manual_coverage = audit.manual_answer_point_coverage.mean()
print('Automated phrase-based coverage:', f'{automatic_coverage:.1%}')
print('Documented manual coverage:', f'{manual_coverage:.1%}')
print('Difference from scoring false negatives/partial credit:', f'{manual_coverage-automatic_coverage:+.1%}')

audit[[
    'question_id', 'expected_source_present', 'answer_point_coverage',
    'manual_answer_point_coverage', 'verdict', 'primary_failure', 'review'
]].style.format({
    'answer_point_coverage': '{:.1%}',
    'manual_answer_point_coverage': '{:.1%}'
})

Automated phrase-based coverage: 58.3%
Documented manual coverage: 63.3%
Difference from scoring false negatives/partial credit: +5.0%


,question_id,expected_source_present,answer_point_coverage,manual_answer_point_coverage,verdict,primary_failure,review
0,Q01,True,0.0%,50.0%,partially supported,generation omission plus phrase-matcher false negative,"The reactive-maintenance distinction is correct, although 'until something breaks' did not match the accepted phrase 'until it breaks'. The preventive description does not clearly state scheduled predetermined intervals and partially resembles predictive maintenance."
1,Q02,True,100.0%,100.0%,supported,none,"The answer covers time, distance, and operating cycles; accepted evidence pages reached the prompt."
2,Q03,True,50.0%,50.0%,partially supported,generation omission,"The answer covers equipment condition but omits the requested sensor/observed data, historical data, and model information even though accepted evidence reached the prompt."
3,Q04,False,0.0%,0.0%,unsupported,retrieval failure,Pages 15-16 containing unnecessary labor/material costs and planned downtime did not reach Top-3. The generated benefits and 'counterintuitive results' do not answer the question and are not adequately grounded.
4,Q05,True,100.0%,100.0%,supported,none,"The answer accurately includes failure prediction/recovery, defect reduction, and early event detection from PDF page 33."
5,Q06,True,50.0%,75.0%,mostly supported,phrase-matcher false negative plus minor omission,"The answer includes advanced warning, redundant inspections, unscheduled maintenance, and test-equipment requirements. The matcher missed 'advanced' versus 'advance'; avoiding catastrophic failure was omitted."
6,Q07,True,100.0%,100.0%,supported,none,The definition correctly states that cascading effects are subsequent damage caused by machine breakdown.
7,Q08,True,100.0%,75.0%,mostly supported,minor generation omission,"The answer correctly states prediction of failure evolution from health information, but does not specify assessed health or diagnostic fault-classifier output."
8,Q09,True,33.3%,33.3%,partially supported,generation omission,"The answer covers measurement across equipment life and maintenance cycles and mentions high enabling cost, but omits the lack of standardized ROI methods and limitations in predictive models, data, methods, and metrics."
9,Q10,True,50.0%,50.0%,partially supported,generation omission,"The answer correctly gives the relative direction of costs, but omits the specific predictive-maintenance profile of low labor and medium parts cost."


In [3]:
print('Manual verdict counts:')
print(audit.verdict.value_counts().to_string())
print('\nPrimary failure counts:')
print(audit.primary_failure.value_counts().to_string())

Manual verdict counts:
verdict
partially supported    4
supported              3
mostly supported       2
unsupported            1

Primary failure counts:
primary_failure
none                                                      3
generation omission                                       3
generation omission plus phrase-matcher false negative    1
retrieval failure                                         1
phrase-matcher false negative plus minor omission         1
minor generation omission                                 1


## Checkpoint 2 — Compare a scoring false negative with a real retrieval failure

Q06 contains the expected concept `advanced warning`, but the accepted literal phrase was `advance warning`. Q04 is not a wording issue: pages 15–16 never reached the prompt, so Qwen received the wrong evidence and produced an unsupported answer.

In [4]:
for question_id in ('Q06', 'Q04'):
    row = audit[audit.question_id == question_id].iloc[0]
    print('\n', '=' * 100)
    print(question_id, '-', row.question)
    print('Expected source reached prompt:', row.expected_source_present)
    print('Automatic coverage:', f'{row.answer_point_coverage:.1%}')
    print('Manual coverage:', f'{row.manual_answer_point_coverage:.1%}')
    print('Primary failure:', row.primary_failure)
    print('Answer:', row.raw_model_answer)
    print('Review:', row.review)


Q06 - What safety and maintainability benefits are associated with prognostics and health management?
Expected source reached prompt: True
Automatic coverage: 50.0%
Manual coverage: 75.0%
Primary failure: phrase-matcher false negative plus minor omission
Answer: Safety and maintainability benefits associated with prognostics and health management include advanced warning of failure, elimination of redundant inspections, minimized unscheduled maintenance, and decreased test equipment requirement.
Review: The answer includes advanced warning, redundant inspections, unscheduled maintenance, and test-equipment requirements. The matcher missed 'advanced' versus 'advance'; avoiding catastrophic failure was omitted.

Q04 - What is a disadvantage of performing excessive preventive maintenance?
Expected source reached prompt: False
Automatic coverage: 0.0%
Manual coverage: 0.0%
Primary failure: retrieval failure
Answer: Based on the provided information, excessive preventive maintenance can le

## Checkpoint 3 — Test the gate on fresh unsupported questions

Five new machine-, personnel-, facility-, and time-specific questions were created after selecting the threshold. Only retrieval is run: if Top-1 is below `0.58`, the production path returns `INSUFFICIENT EVIDENCE` and Qwen is not called.

In [5]:
from src.embed_retrieve import read_jsonl, load_embedding_model, retrieve_top_k
embedding_model = load_embedding_model(PROJECT_ROOT / 'models/all-MiniLM-L6-v2')
chunks = read_jsonl(PROJECT_ROOT / 'data/processed/chunks_128.jsonl')
embeddings = np.load(PROJECT_ROOT / 'data/embeddings/embeddings_128.npy')
holdout_questions = json.loads((PROJECT_ROOT / 'evaluation/holdout_unsupported_questions.json').read_text(encoding='utf-8'))

threshold = 0.58
holdout_rows = []
for item in holdout_questions:
    top_results = retrieve_top_k(item['question'], embedding_model, embeddings, chunks, top_k=3)
    holdout_rows.append({
        'question_id': item['question_id'],
        'question': item['question'],
        'top_similarity': top_results[0]['score'],
        'gate_rejected': top_results[0]['score'] < threshold,
        'top_document': top_results[0]['document'],
        'top_page': top_results[0]['pdf_page'],
        'margin_below_threshold': threshold - top_results[0]['score'],
    })
holdout = pd.DataFrame(holdout_rows)
holdout.style.format({
    'top_similarity': '{:.3f}', 'margin_below_threshold': '{:.3f}'
})

,question_id,question,top_similarity,gate_rejected,top_document,top_page,margin_below_threshold
0,H01,What is the warranty period for a Sandisk SMT placement machine?,0.421,True,NIST.AMS.100-34,34,0.159
1,H02,Which Batu Kawan production line had the most downtime in 2025?,0.412,True,NIST.AMS.100-34,41,0.168
2,H03,What torque setting should be used for a Sandisk assembly fixture?,0.257,True,NIST.AMS.100-2,58,0.323
3,H04,Which technician completed the most maintenance work last month?,0.570,True,NIST.AMS.100-18,15,0.010
4,H05,What was yesterday's defect rate at the Sandisk Batu Kawan facility?,0.482,True,NIST.AMS.100-34,32,0.098


In [6]:
print('Fresh unsupported questions:', len(holdout))
print('Rejected before Qwen call:', int(holdout.gate_rejected.sum()))
print('Holdout rejection rate:', f'{holdout.gate_rejected.mean():.1%}')
print('Highest unsupported similarity:', round(holdout.top_similarity.max(), 3))
print('Narrowest margin below threshold:', round(holdout.margin_below_threshold.min(), 3))

Fresh unsupported questions: 5
Rejected before Qwen call: 5
Holdout rejection rate: 100.0%
Highest unsupported similarity: 0.57
Narrowest margin below threshold: 0.01


## Final evidence-based assessment

| Area | Result | Interpretation |
|---|---:|---|
| Expected source reached grounded Top-3 | 90% | Q04 is the main retrieval failure |
| Automated answer-point coverage | 58.3% | Literal and reproducible, but undercounts paraphrases |
| Documented manual point coverage | 63.3% | Includes partial credit with written rationale |
| At least partially supported answers | 9/10 | Q04 is unacceptable |
| Fresh unsupported gate rejection | 5/5 | Promising but only five holdout questions |
| Narrowest holdout gate margin | 0.010 | Threshold remains fragile |

Keep **grounded Top-3 with a 0.58 MVP gate**, but do not describe it as production-safe. The most valuable next technical improvement is retrieval—query expansion or hybrid lexical/semantic search—because prompt engineering cannot repair missing evidence. The project is now ready for README/CV preparation with these measured limitations retained.